# CS4305 Artificial Intelligence — Lab 1
## Introduction to AI: Chatbot, Agents and History

**How to use:** Run each code cell from top to bottom with **Shift + Enter**. All libraries are already installed in Google Colab — no `pip install` needed.

---
## Program 1: Mini ELIZA Chatbot ("Acting Humanly")
**Aim:** Build a chatbot that *seems* to understand, using only pattern matching and pronoun reflection — like ELIZA (1966).

**How it works:** each rule is a regular-expression pattern with possible replies; `{0}` is filled with the matched words after swapping pronouns (my → your, I → you).

In [ ]:
# ===== Program 1: Mini ELIZA Chatbot =====
import re
import random

# Each rule: (pattern, list of possible responses). {0} is replaced by the matched text.
rules = [
    (r'.*\b(hello|hi|hey|salam|assalam o alaikum)\b.*',
        ["Hello! How are you feeling today?", "Hi there! What would you like to talk about?"]),
    (r'i need (.*)',  ["Why do you need {0}?", "Would it really help you to get {0}?"]),
    (r'i am (.*)',    ["How long have you been {0}?", "Why do you think you are {0}?"]),
    (r'i feel (.*)',  ["Why do you feel {0}?", "Do you often feel {0}?"]),
    (r'.*\b(mother|father|family|brother|sister)\b.*',
        ["Tell me more about your family.", "How do you get along with your {0}?"]),
    (r'.*\bbecause\b.*', ["Is that the real reason?", "What other reasons come to mind?"]),
    (r'.*\?$',        ["Why do you ask that?", "What do you think the answer is?"]),
    (r'(bye|quit|exit)', ["Goodbye! Take care."]),
]
default_replies = ["Please tell me more.", "I see. Go on.", "How does that make you feel?"]

# Pronoun reflection: "my exams" -> "your exams"
reflections = {"i": "you", "me": "you", "my": "your", "am": "are",
               "you": "I", "your": "my", "myself": "yourself"}

def reflect(text):
    return " ".join(reflections.get(word, word) for word in text.split())

def eliza_reply(user_input):
    text = user_input.lower().strip().rstrip(".!")
    for pattern, responses in rules:
        match = re.match(pattern, text)
        if match:
            groups = [reflect(g.strip()) for g in match.groups()]
            return random.choice(responses).format(*groups)
    return random.choice(default_replies)

# --- Demo with fixed inputs ---
for s in ["Hello", "I am worried about my exams", "I need a break",
          "My mother is very strict", "Can you help me?", "The weather is nice today"]:
    print("You  :", s)
    print("ELIZA:", eliza_reply(s))
    print()

**Observe:** ELIZA copies your words back but understands nothing. This is the idea behind Searle's *Chinese Room* argument.

### Talk to ELIZA yourself
Run the cell and type in the box that appears. Type `bye` to stop.

In [ ]:
# ===== Talk to ELIZA yourself =====
print("ELIZA: Hello, I am ELIZA. Type 'bye' to exit.")
while True:
    user = input("You: ")
    print("ELIZA:", eliza_reply(user))
    if user.lower().strip() in ("bye", "quit", "exit"):
        break

---
## Program 2: Simple Reflex Agent in the Vacuum World ("Acting Rationally")
**Aim:** Implement an agent that senses `[location, status]` and chooses an action (`Suck`, `Left`, `Right`) using condition–action rules.

**Performance measure:** +1 point for each clean square at every time step.

In [ ]:
# ===== Program 2: Vacuum-Cleaner World =====
import random

def reflex_vacuum_agent(location, status):
    """Simple reflex agent: acts only on the current percept using condition-action rules."""
    if status == "Dirty":
        return "Suck"
    elif location == "A":
        return "Right"
    else:
        return "Left"

def random_vacuum_agent(location, status):
    """A 'non-intelligent' agent that ignores its percept."""
    return random.choice(["Suck", "Left", "Right"])

def run_vacuum(agent, world=None, location="A", steps=6, verbose=True):
    world = dict(world) if world else {"A": "Dirty", "B": "Dirty"}
    score = 0
    for t in range(1, steps + 1):
        status = world[location]                 # SENSE
        action = agent(location, status)         # THINK
        if verbose:
            print(f"Step {t}: Percept=[{location}, {status}] -> Action={action}")
        if action == "Suck":                     # ACT
            world[location] = "Clean"
        elif action == "Right":
            location = "B"
        elif action == "Left":
            location = "A"
        score += list(world.values()).count("Clean")   # +1 per clean square per step
    if verbose:
        print("Final world:", world, "| Performance score:", score)
    return score

run_vacuum(reflex_vacuum_agent)

### Compare the rational agent with a random agent
Both agents are tested on 500 random starting worlds.

In [ ]:
# ===== Compare rational agent vs random agent =====
random.seed(42)

def average_score(agent, trials=500):
    total = 0
    for _ in range(trials):
        world = {"A": random.choice(["Clean", "Dirty"]), "B": random.choice(["Clean", "Dirty"])}
        location = random.choice(["A", "B"])
        total += run_vacuum(agent, world, location, verbose=False)
    return total / trials

print("Average score - Reflex agent :", average_score(reflex_vacuum_agent))
print("Average score - Random agent :", average_score(random_vacuum_agent))

**Observe:** the reflex agent scores higher because it acts on its percepts. But it keeps moving after both squares are clean — it has no memory.

---
## Program 3: Visualizing the History of AI
**Aim:** Plot the key milestones of AI on a timeline with Matplotlib. Grey bands show the two AI winters.

In [ ]:
# ===== Program 3: AI History Timeline =====
import matplotlib.pyplot as plt

events = [(1943, "McCulloch & Pitts:\nartificial neuron"), (1950, "Turing Test\nproposed"),
          (1956, "Dartmouth: 'AI'\nterm coined"),          (1958, "LISP created\n(McCarthy)"),
          (1966, "ELIZA\nchatbot"),                        (1972, "Prolog\ncreated"),
          (1976, "MYCIN\nexpert system"),                  (1997, "Deep Blue\nbeats Kasparov"),
          (2011, "IBM Watson\nwins Jeopardy!"),            (2012, "Deep learning\nboom (AlexNet)"),
          (2016, "AlphaGo\nbeats Lee Sedol"),              (2022, "ChatGPT &\nLLMs era")]

levels = [0.6, -0.6, 1.0, -1.0]          # stagger labels so they don't overlap
fig, ax = plt.subplots(figsize=(15, 5))
ax.axhline(0, color="black", linewidth=1.5)
for i, (year, label) in enumerate(events):
    h = levels[i % 4]
    ax.vlines(year, 0, h, color="steelblue")
    ax.plot(year, 0, "o", color="darkred")
    ax.text(year, h + (0.05 if h > 0 else -0.05), f"{year}\n{label}",
            ha="center", va="bottom" if h > 0 else "top", fontsize=8)

# Shade the two AI winters
ax.axvspan(1974, 1980, color="lightgrey", alpha=0.6)
ax.text(1977, -1.75, "1st AI\nWinter", ha="center", fontsize=8)
ax.axvspan(1987, 1993, color="lightgrey", alpha=0.6)
ax.text(1990, -1.75, "2nd AI\nWinter", ha="center", fontsize=8)

ax.set_ylim(-2.0, 1.8); ax.set_xlim(1938, 2027); ax.axis("off")
ax.set_title("Milestones in the History of Artificial Intelligence", fontsize=13)
plt.tight_layout()
plt.show()

---
## Lab Tasks
1. Add at least **five new rules** to ELIZA (studies, university, cricket, weather…). Include one rule for a Roman-Urdu greeting.
2. Try to "break" ELIZA with a factual question such as *What is 2+2?* Could it pass the Turing Test?
3. Modify the vacuum agent so it returns `"NoOp"` once it knows both squares are clean (*hint:* use a global dictionary as memory). Does its score change? Why is it no longer a *simple reflex* agent?
4. Add two more AI milestones of your choice to the timeline.